# Thesis Training Runner

Run DQN training on Colab GPU with remote API access.

**Usage:**
1. Connect to a Colab GPU runtime (T4 free tier, or A100 with Pro)
2. Run the cell below -- wait for the output line with URL + token
3. Use the remote API to submit training jobs and download results
4. Output is saved to Google Drive for local download via rclone

In [ ]:
# === Configuration ===
REPO_URL = "https://github.com/toribiodiego/thesis.git"
BRANCH = "main"
API_PORT = 9090
DRIVE_BASE = "/content/drive/MyDrive/thesis-runs"  # where output is copied

# === Setup ===
import subprocess, os, secrets, time, json, signal
from pathlib import Path
from http.server import HTTPServer, BaseHTTPRequestHandler
from threading import Thread

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_BASE, exist_ok=True)

# Clone repo and install dependencies
REPO_DIR = "/content/thesis"
if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# Install GPU requirements
req_file = f"{REPO_DIR}/setup/requirements-gpu.txt"
if not Path(req_file).exists():
    req_file = f"{REPO_DIR}/setup/requirements.txt"
subprocess.run(["pip", "install", "-q", "-r", req_file], check=True)

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU detected"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory // (1024**2)
print(f"[gpu] {gpu_name} ({gpu_mem} MiB)")

# === Remote API Server ===
TOKEN = secrets.token_urlsafe(32)
JOBS_DIR = Path("/tmp/colab-jobs")
JOBS_DIR.mkdir(exist_ok=True)

class Handler(BaseHTTPRequestHandler):
    def log_message(self, *a): pass  # suppress logs

    def _auth(self):
        if self.headers.get("Authorization") != f"Bearer {TOKEN}":
            self.send_response(401); self.end_headers(); return False
        return True

    def _json(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/health":
            self._json(200, {"status": "ok", "gpu": gpu_name}); return
        if not self._auth(): return

        if self.path == "/jobs":
            jobs = []
            for d in sorted(JOBS_DIR.iterdir()):
                if d.is_dir():
                    ec = d / "exit_code"
                    jobs.append({"id": d.name, "status": "complete" if ec.exists() else "running"})
            self._json(200, {"jobs": jobs}); return

        if self.path.startswith("/jobs/"):
            jid = self.path.split("/")[2]
            jdir = JOBS_DIR / jid
            if not jdir.exists():
                self._json(404, {"error": "not found"}); return
            ec_file = jdir / "exit_code"
            if ec_file.exists():
                self._json(200, {
                    "status": "complete",
                    "exit_code": int(ec_file.read_text().strip()),
                    "stdout": (jdir / "stdout").read_text()[-5000:],
                    "stderr": (jdir / "stderr").read_text()[-2000:],
                })
            else:
                # Show tail of stdout for progress
                so = jdir / "stdout"
                tail = so.read_text()[-2000:] if so.exists() else ""
                self._json(200, {"status": "running", "tail": tail})
            return

        if self.path.startswith("/files/"):
            fpath = Path("/content") / self.path[7:]
            if not fpath.exists():
                self._json(404, {"error": "not found"}); return
            self.send_response(200)
            self.send_header("Content-Type", "application/octet-stream")
            self.send_header("Content-Length", str(fpath.stat().st_size))
            self.end_headers()
            with open(fpath, "rb") as f:
                while chunk := f.read(65536):
                    self.wfile.write(chunk)
            return

        self._json(404, {"error": "not found"})

    def do_POST(self):
        if self.path == "/shutdown":
            if not self._auth(): return
            self._json(200, {"status": "shutting down"})
            Thread(target=lambda: (time.sleep(0.5), os._exit(0))).start()
            return

        if self.path != "/exec":
            self._json(404, {"error": "not found"}); return
        if not self._auth(): return

        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        cmd = body.get("cmd", "")
        bg = body.get("background", False)

        if bg:
            jid = secrets.token_hex(4)
            jdir = JOBS_DIR / jid
            jdir.mkdir()
            script = f"#!/bin/bash\ncd {REPO_DIR}\n{cmd}\n"
            (jdir / "run.sh").write_text(script)
            (jdir / "cmd").write_text(cmd)
            with open(jdir / "stdout", "w") as so, open(jdir / "stderr", "w") as se:
                p = subprocess.Popen(
                    ["bash", str(jdir / "run.sh")],
                    stdout=so, stderr=se,
                    start_new_session=True,
                )
            (jdir / "pid").write_text(str(p.pid))
            # Monitor and write exit code when done
            def _wait():
                ec = p.wait()
                (jdir / "exit_code").write_text(str(ec))
            Thread(target=_wait, daemon=True).start()
            self._json(200, {"job_id": jid})
        else:
            try:
                r = subprocess.run(
                    ["bash", "-c", f"cd {REPO_DIR} && {cmd}"],
                    capture_output=True, text=True, timeout=90,
                )
                self._json(200, {"stdout": r.stdout[-5000:], "stderr": r.stderr[-2000:], "exit_code": r.returncode})
            except subprocess.TimeoutExpired:
                self._json(504, {"error": "timeout (90s) -- use background:true for long commands"})

# Kill any prior server on this port
subprocess.run(["fuser", "-k", f"{API_PORT}/tcp"], capture_output=True)
time.sleep(0.5)

server = HTTPServer(("", API_PORT), Handler)
server.allow_reuse_address = True
Thread(target=server.serve_forever, daemon=True).start()
print(f"[api] Server listening on port {API_PORT}")

# === Cloudflared Tunnel ===
subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{API_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(5)

# Extract tunnel URL from stderr
import select
url = None
for _ in range(20):
    if select.select([tunnel.stderr], [], [], 1.0)[0]:
        line = tunnel.stderr.readline().decode()
        if "trycloudflare.com" in line:
            import re
            m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
            if m: url = m.group(0); break

if url:
    print(f"\n{'='*60}")
    print(f"TUNNEL_URL={url}")
    print(f"TOKEN={TOKEN}")
    print(f"DRIVE_BASE={DRIVE_BASE}")
    print(f"{'='*60}")
    print(f"\nHealth check: curl -s {url}/health")
else:
    print("[error] Could not extract tunnel URL. Check cloudflared output.")

# Keep cell alive
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    print("[shutdown] Stopping server")
    server.shutdown()